# Gravity layout: what the knob does

Gravity generalizes the `phi` value moves. For a mark anchored at its category line,
a candidate position at offset `doff` and value move `dval` costs

    C_g = doff^2 * (1 + g*kappa*doff^2) + phi*dval^2 - g*beta*rho

where `rho` is the smoothed density of already-placed marks, each weighted by its own
distance from this mark's category line. `g = 0` is exactly `phi`. The growth term
makes far-out marks trade value moves more readily; the basin rewards landing near
density. Marks whose anchor is free never move (the gate), so sparse plots are untouched.

This notebook exercises the engine at a few `g` values on the same data. It is a
correctness reference, not a tuning study: parameters are at their placeholder
defaults, sizes are fixed so the effect of `g` is visible, and timings are printed.
See `.claude/BEESWARM_GRAVITY_RECORD.md` for the design, the anchors, and the levers.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

from idd_figures.beeswarm_core import Gravity, has_fast_backend
from idd_figures.idd_beeswarm import SCATTER_LW, position_all_points

print("compiled kernel in use:", has_fast_backend())

rng = np.random.default_rng(11)
N_PER = 28
x = np.repeat([0.0, 1.0], N_PER)
y = np.concatenate([rng.normal(0.0, 1.0, N_PER), rng.normal(0.3, 0.8, N_PER)])
S = 90.0      # fixed marker size: the same dots at every g
GAP = 0.1
PHI = 2.0


def swarm_axes(ax):
    yd = y.max() - y.min()
    ax.set_xlim(-0.9, 1.9)
    ax.set_ylim(y.min() - 0.25 * yd, y.max() + 0.25 * yd)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["A", "B"])
    for xp in (0.0, 1.0):
        ax.axvline(xp, color="gray", lw=0.5, alpha=0.4)


def draw(ax, gravity, title):
    swarm_axes(ax)
    t0 = time.perf_counter()
    res, extent = position_all_points(
        x, y, S, GAP, plt.gcf(), ax, process_order="spine-drop", phi=PHI, gravity=gravity
    )
    dt = time.perf_counter() - t0
    ax.scatter(res["xnew"], res["ynew"], s=S, facecolors="#1f6f8b", edgecolors=None, linewidths=SCATTER_LW)
    shift = np.abs(res["xnew"] - x).mean()
    dval = np.abs(res["ynew"] - y).mean()
    ax.set_title(f"{title}\nmean |offset| {shift:.3f}, mean |value move| {dval:.3f}, {dt:.2f} s", fontsize=9)
    return res

## Sweep `g` at fixed marker size

Same data, same size, spine-drop, `phi = 2`. Left to right: `g = 0` (exactly phi+drop),
then increasing gravity. Watch the swarm narrow (smaller mean offset) as marks accept
larger value moves, and watch marks gather where others already sit.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
for ax, g in zip(axes, [0.0, 0.5, 2.0, 8.0]):
    draw(ax, Gravity(g), f"g = {g}")
fig.tight_layout()

## The two terms separately at `g = 2`

Growth only (`beta = 0`): the offset price grows with distance from the line, so far
marks move in value instead. Basin only (`kappa = 0`): marks are pulled toward existing
density with no extra offset price. Both on: the default.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
draw(axes[0], Gravity(2.0, beta=0.0), "g = 2, growth only (beta = 0)")
draw(axes[1], Gravity(2.0, kappa=0.0), "g = 2, basin only (kappa = 0)")
draw(axes[2], Gravity(2.0), "g = 2, both (defaults)")
fig.tight_layout()

## The anchor: `g = 0` against phi+drop

With the sampler off (`exhaustive=False`) gravity at `g = 0` is phi+drop bit for bit.
With the sampler on it is never worse per placement and occasionally strictly better,
because phi's analytic candidate set is complete only up to a rare second local minimum
on a partially covered circle; the greedy then cascades, so whole-layout differences can be
larger than the per-placement gain. All three panels sit in ONE figure on purpose: the
marker size is fixed in points, so the data-to-pixel scale, and therefore the packing,
depends on panel geometry. Compare layouts only within a figure.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.5))
ref = draw(axes[0], None, "phi = 2, spine-drop (no gravity)")
an = draw(axes[1], Gravity(0.0, exhaustive=False), "gravity g = 0, analytic mode")
ex = draw(axes[2], Gravity(0.0), "gravity g = 0, exhaustive")


def highlight(ax, other, ref, label):
    # red ring on every dot whose landing differs from the phi panel
    moved = np.any(np.abs(other[["xnew", "ynew"]].to_numpy() - ref[["xnew", "ynew"]].to_numpy()) > 1e-9, axis=1)
    ax.scatter(other["xnew"][moved], other["ynew"][moved], s=S * 3.2, facecolors="none", edgecolors="#c0504d", linewidths=1.8)
    d = np.abs(other[["xnew", "ynew"]].to_numpy() - ref[["xnew", "ynew"]].to_numpy()).max()
    print(f"{label}: {int(moved.sum())} of {len(ref)} dots differ from phi; max |difference| {d:.3g}")
    return moved


highlight(axes[1], an, ref, "analytic g=0  (expect 0: bit-exact)")
highlight(axes[2], ex, ref, "exhaustive g=0 (sampler found cheaper landings phi missed, then the greedy cascaded)")
fig.tight_layout()

## Notes

- Positions come from the C kernel when it is built (`has_fast_backend()`), else from the
  Python reference; results agree to 1e-9 (gravity) and are identical for the other engines.
- Everything here is at fixed `s` (points). Panel geometry changes the packing; compare within a figure.
- Everything here is at fixed `s`. Auto-sizing (`find_optimal_s`) works with gravity too but
  multiplies the layout cost by the search's iteration count.
- Tuning `kappa`, `beta`, `sigma`, `lam`, and judging what looks right, is the exploration
  session's work, against this reference.